# Validation design

The model trains on old vintages and is scored on newer ones. That choice only matters if the
population changes over time, and it costs something if it does. This notebook checks both: what
moved between the training and validation periods, and what evaluating out-of-time is worth.

In short:

- To see what moved, PSI measures each feature's drift one at a time and adversarial validation
  measures whether the two periods are separable at all. The borrower features are stable, every
  one below 0.04 and only 0.63 separable on their own, while `int_rate` alone takes that to 0.96.
- To see what it costs, we run a random split against the temporal one and isolate the effect by
  scoring both training sets on the same loans. The optimism a shuffle buys is real but small,
  about 0.003 of ROC AUC, stable across seeds.

Everything here uses the union feature set from notebook 21: the borrower data together with
Lending Club's verdict, scored with LightGBM, the strongest combination there. Drift and
out-of-time cost only matter for the model that would actually be deployed, so the analysis
follows that one rather than repeating across feature sets. The one place it splits the two apart
is the adversarial check, which is run with and without the verdict to see where the drift sits.

In [16]:
import pandas as pd

from credit_risk.data import load_loans
from credit_risk.split import out_of_time_split, random_split
from credit_risk.model import (
    build_lgbm,
    LC_VERDICT_NUMERIC, LC_VERDICT_CATEGORICAL,
    UNDERWRITER_NUMERIC, UNDERWRITER_CATEGORICAL,
)
from credit_risk.evaluate import discrimination_metrics
from credit_risk.drift import population_stability_index, adversarial_validation

TARGET = "target_bad"

# The union set, which performed best in notebook 21.
NUMERIC = UNDERWRITER_NUMERIC + LC_VERDICT_NUMERIC
CATEGORICAL = UNDERWRITER_CATEGORICAL + LC_VERDICT_CATEGORICAL

df = load_loans()
train_temp, val_temp, _ = out_of_time_split(df)

print(f"train {len(train_temp)}  validation {len(val_temp)}")

train 375212  validation 154703


## What moved

Two views of the same question. PSI takes one feature at a time: it fixes bins on the training
sample and sums how far the validation shares diverge, read as stable below 0.10 and a large
shift above 0.25. Adversarial validation instead labels training rows against validation rows and
tries to tell them apart from the features, so it also catches joint movement that leaves every
marginal intact.

In [17]:
# Deciles only mean something with enough distinct values, so the near-constant derogatory
# counts are left out rather than reported as a number that cannot be read.
enough_values = [col for col in NUMERIC if train_temp[col].nunique() >= 20]
skipped = sorted(set(NUMERIC) - set(enough_values))
print("skipped, too few distinct values:", skipped)

psi_by_feature = {}
for col in enough_values:
    expected = train_temp[col].dropna()
    actual = val_temp[col].dropna()
    psi_by_feature[col] = population_stability_index(expected, actual)

psi = pd.Series(psi_by_feature).sort_values(ascending=False)
psi.round(3)

skipped, too few distinct values: ['acc_now_delinq', 'chargeoff_within_12_mths', 'collections_12_mths_ex_med', 'inq_last_6mths', 'pub_rec_bankruptcies']


int_rate                  0.155
dti                       0.033
credit_history_months     0.014
open_acc                  0.014
revol_util                0.013
active_acct_ratio         0.012
pub_rec                   0.010
fico_range_low            0.008
loan_amnt                 0.007
mths_since_last_delinq    0.004
revol_bal                 0.004
delinq_2yrs               0.003
annual_inc                0.003
loan_to_income            0.002
total_acc                 0.001
tax_liens                 0.000
delinq_amnt               0.000
dtype: float64

In [ ]:
# Run it twice, with and without Lending Club's verdict, to see whether the separability comes
# from the borrower data or from the pricing.
with_verdict = adversarial_validation(train_temp, val_temp, NUMERIC, CATEGORICAL)
borrower_only = adversarial_validation(
    train_temp, val_temp, UNDERWRITER_NUMERIC, UNDERWRITER_CATEGORICAL
)

print(f"with the LC verdict: AUC {with_verdict['roc_auc']:.3f}")
print(f"borrower data only:  AUC {borrower_only['roc_auc']:.3f}")
print()
print(with_verdict["importances"].head(5))

The borrower features are stable: every one below 0.04 on PSI, `fico_range_low` among the most
stable of all, and a model given only borrower data separates the two periods at 0.63, which is
mild. What moved is Lending Club's pricing. `int_rate` is the single feature above 0.10, and it
alone takes the separability to 0.96.

Two things worth keeping. PSI found almost nothing feature by feature, yet borrower data still
separates the periods at 0.63, which is joint movement a marginal view cannot see. And training
spans eight years against seven months of validation, so the long drift across vintages in
`sql/11_monthly_originations.sql` sits inside the training window rather than between the two
sets.

## What it costs

We compare the temporal split against a random one to measure how much optimism a shuffle buys.
A random split lets the model train on loans from the same periods it is then scored on, which is
exactly what the out-of-time setup withholds, so the gap between the two is the price of
evaluating honestly.

In [ ]:
def fit_and_score(train, val):
    # Fit on training data and calculate validation metrics.
    cols = NUMERIC + CATEGORICAL

    pipe = build_lgbm(NUMERIC, CATEGORICAL)
    pipe.fit(train[cols], train[TARGET])

    proba = pipe.predict_proba(val[cols])[:, 1]
    y = val[TARGET].values

    metrics = discrimination_metrics(y, proba)
    return {
        "roc_auc": metrics["roc_auc"],
        "pr_auc": metrics["pr_auc"],
        "brier": metrics["brier"],
    }


train_rand, val_rand, _ = random_split(df)

temporal_metrics = fit_and_score(train_temp, val_temp)
random_metrics = fit_and_score(train_rand, val_rand)

pd.DataFrame({"temporal": temporal_metrics, "random": random_metrics}).T.round(4)

The borrower features barely moved above, so a shuffle should buy little optimism, and the two
scores are indeed close, the random one slightly lower rather than higher. This head-to-head
still cannot confirm it cleanly, since the two splits score on different populations: a
seven-month slice of 2015 against a sample of the whole history, so any gap also mixes in the
loans being scored.

Holding the evaluation set fixed isolates the split. Both models below are scored on the same
validation slice and only the training composition changes: the past alone, against a same-size
sample drawn from every vintage except the ones being scored. The second is what a random split
hands the model and what production never can.

In [ ]:
# The mixed sample can draw from any vintage except the loans being scored.
pool = df.drop(index=val_temp.index)
n_train = len(train_temp)

# How much of such a sample was issued after the loans it gets scored on?
last_scored = val_temp["issue_month"].max()
example = pool.sample(n=n_train, random_state=0)
from_the_future = (example["issue_month"] > last_scored).mean()
print(f"share of the mixed sample issued after the validation period: {from_the_future:.1%}")

results = {}
results["past only"] = fit_and_score(train_temp, val_temp)

for seed in [0, 1, 2]:
    mixed = pool.sample(n=n_train, random_state=seed)
    results[f"period mixed (seed {seed})"] = fit_and_score(mixed, val_temp)

pd.DataFrame(results).T.round(4)

Training on a period-mixed sample, about a third of it issued after the loans being scored, is
worth roughly 0.002 to 0.003 of ROC AUC, and the gap holds across seeds. Small: `int_rate` drifts
in level, but not in a way that hurts the ranking.

## Conclusions

Between the two periods the borrowers barely changed. Lending Club's pricing did: `int_rate`
drifted, but in level, not in a way that hurt its ranking. That is why training without the later vintages costs so little. Scored on the same loans, the
cost is about 0.003 of ROC AUC.

The caveat comes from the same finding. The model leans on `int_rate` more than on any other
feature, and `int_rate` is the one that moves. Performance held here, but that pairing is what to
monitor.

The rationale for using a temporal split remains. A random split evaluates the model on loans from the same periods used for training, rather than on future data, which is the setting encountered in deployment.